# Multi-Model Forecast Comparison

Run multiple AI weather models on the same date and compare their skill against ERA5 reanalysis.

**Extra installs:**
```
uv add earth2studio --extra sfno --extra pangu --extra fuxi --extra statistics
```

In [ ]:
import os
from datetime import datetime

import torch
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import earth2studio.run as run
from earth2studio.models.px import FCN, GraphCastOperational, SFNO, Pangu6, FuXi
from earth2studio.data import GFS, WB2ERA5
from earth2studio.io import ZarrBackend

In [ ]:
CONFIG = {
    "forecast_date": "2023-01-08",  # must be <= 2023 for ERA5 verification
    "nsteps": 10,
    "output_root": "outputs/model_comparison",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_root"], exist_ok=True)
print(f"Device: {CONFIG['device']}")

## Load Models and Data

In [ ]:
# Load all models — adjust this list based on your GPU VRAM
# GraphCast and SFNO need ~16GB+, Pangu and FuXi are lighter
MODELS = {}

print("Loading FCN...")
MODELS["FCN"] = FCN.load_model(FCN.load_default_package())

# Uncomment models your GPU can handle:
# print("Loading GraphCast...")
# MODELS["GraphCast"] = GraphCastOperational.load_model(GraphCastOperational.load_default_package())

# print("Loading SFNO...")
# MODELS["SFNO"] = SFNO.load_model(SFNO.load_default_package())

print("Loading Pangu6...")
MODELS["Pangu6"] = Pangu6.load_model(Pangu6.load_default_package())

# print("Loading FuXi...")
# MODELS["FuXi"] = FuXi.load_model(FuXi.load_default_package())

gfs_data = GFS()
era5_data = WB2ERA5(cache=True, verbose=True)

print(f"Loaded {len(MODELS)} models: {list(MODELS.keys())}")

## Run Forecasts

In [ ]:
zarr_paths = {}

for name, model in MODELS.items():
    path = f"{CONFIG['output_root']}/{name.lower()}_forecast.zarr"
    io = ZarrBackend(path, backend_kwargs={"overwrite": True})
    print(f"Running {name}...")
    io = run.deterministic([CONFIG["forecast_date"]], CONFIG["nsteps"], model, gfs_data, io)
    zarr_paths[name] = path
    print(f"  {name} complete.")

print(f"\nAll forecasts stored: {list(zarr_paths.keys())}")

## Fetch ERA5 Verification Data

In [ ]:
# Open the first model's zarr to get the grid and lead times
first_model = list(zarr_paths.keys())[0]
ds_ref = xr.open_zarr(zarr_paths[first_model])
lats = ds_ref["lat"].values
lons = ds_ref["lon"].values
n_steps = ds_ref.sizes["lead_time"]

# Fetch ERA5 t2m at each valid time
from earth2studio.data import fetch_data, prep_data_array
from earth2studio.utils.time import to_time_array
from datetime import timedelta

start = datetime.strptime(CONFIG["forecast_date"], "%Y-%m-%d")
era5_t2m = []
for s in range(n_steps):
    valid_time = start + timedelta(hours=6 * s)
    da = era5_data(valid_time, ["t2m"])
    era5_t2m.append(da.values.squeeze())

era5_t2m = np.stack(era5_t2m, axis=0)  # (lead_time, lat, lon)
print(f"ERA5 verification shape: {era5_t2m.shape}")

## Side-by-Side T2M Comparison

In [ ]:
step = 4  # +24h

model_names = list(zarr_paths.keys())
n_models = len(model_names)
ncols = min(n_models + 1, 3)
nrows = (n_models + 1 + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows),
    subplot_kw={"projection": ccrs.Robinson()})
axes = np.atleast_2d(axes).flat

# ERA5 truth
era5_c = era5_t2m[step] - 273.15
im = axes[0].pcolormesh(lons, lats, era5_c, transform=ccrs.PlateCarree(),
                         cmap="RdBu_r", vmin=-40, vmax=40, shading="auto")
axes[0].coastlines()
axes[0].set_title("ERA5 (Truth)", fontsize=12)

# Each model
for i, name in enumerate(model_names):
    ds = xr.open_zarr(zarr_paths[name])
    t2m_c = ds["t2m"].isel(time=0, lead_time=step).values - 273.15
    axes[i + 1].pcolormesh(lons, lats, t2m_c, transform=ccrs.PlateCarree(),
                            cmap="RdBu_r", vmin=-40, vmax=40, shading="auto")
    axes[i + 1].coastlines()
    axes[i + 1].set_title(name, fontsize=12)

# Hide unused axes
for j in range(n_models + 1, len(list(np.atleast_2d(fig.axes).flat))):
    fig.delaxes(list(fig.axes)[j])

fig.suptitle(f"T2M Comparison — +{step*6}h  |  Init: {CONFIG['forecast_date']}", fontsize=14)
cbar = fig.colorbar(im, ax=list(fig.axes), orientation="horizontal",
                    pad=0.06, shrink=0.5, label="2m Temperature (°C)")
plt.savefig(f"{CONFIG['output_root']}/t2m_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Error Maps (Model - ERA5)

In [ ]:
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6),
    subplot_kw={"projection": ccrs.Robinson()})
if n_models == 1:
    axes = [axes]

for i, name in enumerate(model_names):
    ds = xr.open_zarr(zarr_paths[name])
    t2m_model = ds["t2m"].isel(time=0, lead_time=step).values
    error = t2m_model - era5_t2m[step]  # in Kelvin (same as Celsius diff)

    im = axes[i].pcolormesh(lons, lats, error, transform=ccrs.PlateCarree(),
                             cmap="RdBu_r", vmin=-5, vmax=5, shading="auto")
    axes[i].coastlines()
    axes[i].set_title(f"{name} Error", fontsize=12)

fig.suptitle(f"T2M Forecast Error — +{step*6}h", fontsize=14)
cbar = fig.colorbar(im, ax=axes, orientation="horizontal",
                    pad=0.06, shrink=0.5, label="Error (K)")
plt.savefig(f"{CONFIG['output_root']}/error_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## RMSE vs Lead Time

In [ ]:
def compute_rmse(forecast, truth):
    """Global RMSE between two 2D fields."""
    return np.sqrt(np.nanmean((forecast - truth) ** 2))


rmse_results = {name: [] for name in model_names}

for s in range(n_steps):
    for name in model_names:
        ds = xr.open_zarr(zarr_paths[name])
        t2m_model = ds["t2m"].isel(time=0, lead_time=s).values
        rmse = compute_rmse(t2m_model, era5_t2m[s])
        rmse_results[name].append(rmse)

hours = np.arange(n_steps) * 6

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set1(np.linspace(0, 1, len(model_names)))
for i, name in enumerate(model_names):
    ax.plot(hours, rmse_results[name], marker="o", markersize=4,
            label=name, color=colors[i], linewidth=2)

ax.set_xlabel("Lead Time (hours)")
ax.set_ylabel("RMSE (K)")
ax.set_title(f"T2M RMSE vs Lead Time  |  Init: {CONFIG['forecast_date']}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/rmse_vs_leadtime.png", dpi=150, bbox_inches="tight")
plt.show()

## Multi-Variable Comparison at +48h

In [ ]:
variables = ["t2m", "z500", "u10m"]
step_48h = 8  # +48h

# Compute RMSE for each model x variable
results = {name: {} for name in model_names}

for name in model_names:
    ds = xr.open_zarr(zarr_paths[name])
    for var in variables:
        if var in ds.data_vars:
            model_field = ds[var].isel(time=0, lead_time=step_48h).values
            # Fetch ERA5 for this variable
            era5_var = era5_data(
                start + timedelta(hours=6 * step_48h), [var]
            ).values.squeeze()
            results[name][var] = compute_rmse(model_field, era5_var)
        else:
            results[name][var] = np.nan

# Grouped bar chart
x = np.arange(len(variables))
width = 0.8 / len(model_names)

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(model_names):
    vals = [results[name].get(v, np.nan) for v in variables]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])

ax.set_xlabel("Variable")
ax.set_ylabel("RMSE")
ax.set_title(f"RMSE at +48h by Variable  |  Init: {CONFIG['forecast_date']}")
ax.set_xticks(x + width * (len(model_names) - 1) / 2)
ax.set_xticklabels(variables)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/multi_variable_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary Table

In [ ]:
import pandas as pd

rows = []
for name in model_names:
    row = {"Model": name}
    for var in variables:
        row[f"{var} RMSE"] = f"{results[name].get(var, np.nan):.3f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())
df